In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as cx
import datania

# Load the Quarterly Labour Force Survey data
qlfs_csv = datania.generate_labour_force_survey(n_persons=800, seed=2026)
qlfs = pd.read_csv(qlfs_csv)

print(f"QLFS records loaded: {len(qlfs)}")
print(f"\nEmployment status breakdown:")
print(qlfs['employment_status'].value_counts())
print(f"\nProvinces in data: {qlfs['province'].unique()}")
qlfs.head()
# Task 1: Create respondent locations as GeoDataFrame
# The QLFS doesn't have exact coordinates, so we'll simulate them
# based on province centroids with some random scatter

# Province approximate centroids (for simulation)
province_centroids = {
    'NTH': (-11.0, 31.0),
    'STH': (-16.5, 30.0),
    'EST': (-13.0, 34.5),
    'WST': (-14.5, 24.5),
    'CTR': (-13.5, 30.0),
    'LKS': (-11.5, 34.5)
}

# Add simulated coordinates based on province
np.random.seed(2026)
qlfs['latitude'] = qlfs['province'].map(lambda p: province_centroids.get(p, (-14, 28))[0])
qlfs['longitude'] = qlfs['province'].map(lambda p: province_centroids.get(p, (-14, 28))[1])

# Add random scatter within provinces
qlfs['latitude'] += np.random.uniform(-1.5, 1.5, len(qlfs))
qlfs['longitude'] += np.random.uniform(-1.5, 1.5, len(qlfs))

print("Coordinates added to QLFS data")
print(f"Latitude range: {qlfs['latitude'].min():.2f} to {qlfs['latitude'].max():.2f}")
print(f"Longitude range: {qlfs['longitude'].min():.2f} to {qlfs['longitude'].max():.2f}")
# YOUR CODE HERE: Convert to GeoDataFrame
geometry = # Create Point objects from longitude and latitude

qlfs_gdf = gpd.GeoDataFrame(
    qlfs,
    geometry=geometry,
    crs="EPSG:4326"
)

print(f"GeoDataFrame created with {len(qlfs_gdf)} records")
print(f"CRS: {qlfs_gdf.crs}")
qlfs_gdf[['person_id', 'province', 'employment_status', 'geometry']].head()
# Task 2: Create province boundary polygons
provinces_data = {
    'province_code': ['NTH', 'STH', 'EST', 'WST', 'CTR', 'LKS'],
    'province_name': ['Northern Province', 'Southern Province', 'Eastern Province',
                      'Western Province', 'Central Province', 'Lakeside Province'],
    'geometry': [
        box(29, -13, 33, -9),   # Northern - adjusted bounds
        box(27, -19, 33, -14),  # Southern
        box(32, -15, 37, -10),  # Eastern
        box(21, -17, 27, -11),  # Western
        box(27, -16, 33, -11),  # Central
        box(32, -13, 37, -9),   # Lakeside
    ]
}

provinces_gdf = gpd.GeoDataFrame(provinces_data, crs="EPSG:4326")
print("Province boundaries created")
provinces_gdf
# Task 3: Perform spatial join to verify/tag provinces
# This demonstrates data quality checking - do the recorded provinces
# match where the GPS coordinates actually fall?

# YOUR CODE HERE: Spatial join
qlfs_tagged = gpd.sjoin(
    # Left: QLFS points
    # Right: Province polygons
    # how: "left"
    # predicate: "within"
)

print(f"Records after spatial join: {len(qlfs_tagged)}")

# Compare recorded province vs. spatially-determined province
qlfs_tagged['province_match'] = qlfs_tagged['province'] == qlfs_tagged['province_code']
match_rate = qlfs_tagged['province_match'].mean() * 100
print(f"\nProvince match rate: {match_rate:.1f}%")
print("(Some mismatch is expected due to our simplified province boundaries)")
# Task 4: Calculate employment statistics by province
# YOUR CODE HERE: Group by province and calculate employment rates

# Count employment status by province
employment_by_province = qlfs_tagged.groupby(['province_name', 'employment_status']).size().unstack(fill_value=0)

print("Employment counts by province:")
print(employment_by_province)

# Calculate unemployment rate by province
# Unemployment rate = Unemployed / (Employed + Unemployed) * 100
if 'Employed' in employment_by_province.columns and 'Unemployed' in employment_by_province.columns:
    employment_by_province['unemployment_rate'] = (
        employment_by_province['Unemployed'] /
        (employment_by_province['Employed'] + employment_by_province['Unemployed']) * 100
    ).round(1)
    print("\nUnemployment rates by province:")
    print(employment_by_province['unemployment_rate'])
# Task 5: Create publication-ready multi-layer map

# Convert to Web Mercator for basemap
qlfs_web = qlfs_tagged.to_crs(epsg=3857)
provinces_web = provinces_gdf.to_crs(epsg=3857)

# Create figure
fig, ax = plt.subplots(figsize=(16, 14))

# YOUR CODE HERE: Plot the layers

# Layer 1: Province boundaries
provinces_web.plot(
    ax=ax,
    facecolor='none',
    edgecolor='navy',
    linewidth=2.5,
    zorder=2
)

# Layer 2: QLFS respondents colored by employment status
# Use different colors for Employed, Unemployed, Not in labour force
qlfs_web.plot(
    ax=ax,
    column='employment_status',
    cmap='RdYlGn',  # Red-Yellow-Green: intuitive for employment
    markersize=40,
    alpha=0.7,
    legend=True,
    legend_kwds={'title': 'Employment Status', 'loc': 'lower right'},
    zorder=3
)

# Layer 3: Basemap
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zorder=1)

# Title and cleanup
ax.set_title('Quarterly Labour Force Survey - Respondent Locations\nQ1 2026, Republic of Datania',
             fontsize=18, fontweight='bold', pad=20)
ax.set_axis_off()

plt.tight_layout()
plt.show()
# Task 6 (Bonus): Add province labels to the map

fig, ax = plt.subplots(figsize=(16, 14))

# Recreate the layers
provinces_web.plot(ax=ax, facecolor='none', edgecolor='navy', linewidth=2.5, zorder=2)
qlfs_web.plot(ax=ax, column='employment_status', cmap='RdYlGn', markersize=40,
              alpha=0.7, legend=True, legend_kwds={'title': 'Employment Status', 'loc': 'lower right'}, zorder=3)
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zorder=1)

# YOUR CODE HERE: Add province labels at centroids
for idx, row in provinces_web.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        row['province_name'].replace(' Province', ''),  # Shorter labels
        xy=(centroid.x, centroid.y),
        ha='center',
        va='center',
        fontsize=11,
        fontweight='bold',
        color='darkblue',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7)
    )

ax.set_title('QLFS Q1 2026 - Employment Status by Location\nPrepared for Cabinet Meeting',
             fontsize=18, fontweight='bold', pad=20)
ax.set_axis_off()

# Save for the Minister
fig.savefig('qlfs_employment_map.png', dpi=300, bbox_inches='tight')
print("Map saved as 'qlfs_employment_map.png'")

plt.tight_layout()
plt.show()